In [ ]:
!pip install -q kagglehub numpy pandas scipy scikit-learn joblib catboost xgboost lightgbm tqdm tensorflow

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import re
import json
import zipfile
import pickle
import shutil
import joblib
import random
import warnings
from pathlib import Path
from collections import Counter

import kagglehub
import numpy as np
import pandas as pd

from scipy.stats import skew, kurtosis, iqr
from scipy.signal import welch

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow.keras import layers, Model

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

BASE_DIR = Path("/content")
WORK_DIR = BASE_DIR / "wesad_fast_work"
EXTRACT_DIR = WORK_DIR / "extracted"
OUTPUT_DIR = BASE_DIR / "wesad_fast_output"
MODELS_DIR = OUTPUT_DIR / "models"
REPORTS_DIR = OUTPUT_DIR / "reports"

for d in [WORK_DIR, EXTRACT_DIR, OUTPUT_DIR, MODELS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FEATURE_MODE = "wrist_chest"
SPLIT_MODE = "group_split"

WINDOW_SEC = 60
STEP_SEC = 15
RANDOM_STATE = 42

# Fast settings
RUN_ML_MODELS = True
RUN_DEEP_MODELS = True
RUN_TRANSFORMER = True
RUN_SVM = True
RUN_ENSEMBLE = True

DL_EPOCHS = 70
DL_PATIENCE = 8
DL_BATCH_SIZE = 64

THRESHOLD_MIN = 0.10
THRESHOLD_MAX = 0.90
THRESHOLD_STEP = 0.01

FS_WRIST = {
    "BVP": 64,
    "EDA": 4,
    "TEMP": 4,
    "ACC": 32
}

FS_CHEST = {
    "ACC": 700,
    "ECG": 700,
    "EDA": 700,
    "EMG": 700,
    "RESP": 700,
    "TEMP": 700
}

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

print("Fast WESAD trainer ready.")
print("Feature mode:", FEATURE_MODE)
print("Split mode:", SPLIT_MODE)

Fast WESAD trainer ready.
Feature mode: wrist_chest
Split mode: group_split


In [ ]:
print("Downloading WESAD dataset using KaggleHub...")

dataset_path = kagglehub.dataset_download(
    "orvile/wesad-wearable-stress-affect-detection-dataset"
)

dataset_path = Path(dataset_path)

print("Dataset path:")
print(dataset_path)


def extract_zip_file(zip_path, target_dir):
    target_dir.mkdir(parents=True, exist_ok=True)

    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(target_dir)

        print("Extracted:", zip_path.name)
        return True

    except Exception as e:
        print("Failed to extract:", zip_path)
        print("Error:", e)
        return False


def extract_all_zips_recursively(source_dir, output_dir):
    already_extracted = set()

    while True:
        zip_files = list(source_dir.rglob("*.zip")) + list(output_dir.rglob("*.zip"))
        zip_files = list(set(zip_files))

        new_zip_found = False

        for zip_file in zip_files:
            if str(zip_file) in already_extracted:
                continue

            safe_name = zip_file.stem.replace(" ", "_").replace("+", "_")
            target = output_dir / safe_name

            if target.exists() and any(target.iterdir()):
                already_extracted.add(str(zip_file))
                continue

            ok = extract_zip_file(zip_file, target)
            already_extracted.add(str(zip_file))

            if ok:
                new_zip_found = True

        if not new_zip_found:
            break


extract_all_zips_recursively(dataset_path, EXTRACT_DIR)

subject_files = []

for root in [dataset_path, EXTRACT_DIR]:
    subject_files.extend(list(root.rglob("S*.pkl")))

subject_files = sorted(set(subject_files))
subject_files = [p for p in subject_files if re.match(r"^S\d+\.pkl$", p.name)]

print("Found subject files:", len(subject_files))

for p in subject_files:
    print(p)

if len(subject_files) == 0:
    raise RuntimeError("No WESAD subject .pkl files found.")

100%|██████████| 2.43G/2.43G [01:01<00:00, 42.2MB/s]

Extracting files...


Dataset path:
/root/.cache/kagglehub/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/versions/1
Found subject files: 15
/root/.cache/kagglehub/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/versions/1/WESAD/S10/S10.pkl
/root/.cache/kagglehub/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/versions/1/WESAD/S11/S11.pkl
/root/.cache/kagglehub/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/versions/1/WESAD/S13/S13.pkl
/root/.cache/kagglehub/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/versions/1/WESAD/S14/S14.pkl
/root/.cache/kagglehub/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/versions/1/WESAD/S15/S15.pkl
/root/.cache/kagglehub/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/versions/1/WESAD/S16/S16.pkl
/root/.cache/kagglehub/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/versions/1/WESAD/S17/S17.pkl
/root/.cache/kagglehub/datasets/orvile/wesad-wearable-stress

In [ ]:
def load_subject(path):
    with open(path, "rb") as f:
        return pickle.load(f, encoding="latin1")


def safe_array(x):
    arr = np.asarray(x, dtype=np.float64)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return arr


def clean_number(v):
    try:
        v = float(v)
    except Exception:
        return 0.0

    if not np.isfinite(v):
        return 0.0

    return v


def zero_crossing_rate(x):
    x = np.asarray(x).reshape(-1)

    if len(x) < 2:
        return 0.0

    return float(np.mean(np.diff(np.signbit(x))))


def spectral_features(x, fs, prefix):
    x = safe_array(x).reshape(-1)

    if len(x) < 8:
        return {
            f"{prefix}_dom_freq": 0.0,
            f"{prefix}_spec_power": 0.0,
            f"{prefix}_spec_entropy": 0.0,
        }

    try:
        freqs, power = welch(x, fs=fs, nperseg=min(256, len(x)))
        total_power = np.sum(power)

        if total_power <= 1e-12:
            return {
                f"{prefix}_dom_freq": 0.0,
                f"{prefix}_spec_power": 0.0,
                f"{prefix}_spec_entropy": 0.0,
            }

        p_norm = power / total_power
        entropy = -np.sum(p_norm * np.log(p_norm + 1e-12))

        return {
            f"{prefix}_dom_freq": clean_number(freqs[np.argmax(power)]),
            f"{prefix}_spec_power": clean_number(total_power),
            f"{prefix}_spec_entropy": clean_number(entropy),
        }

    except Exception:
        return {
            f"{prefix}_dom_freq": 0.0,
            f"{prefix}_spec_power": 0.0,
            f"{prefix}_spec_entropy": 0.0,
        }


def stats_features(values, prefix, fs=None):
    x = safe_array(values).reshape(-1)

    if len(x) == 0:
        base = {
            f"{prefix}_mean": 0.0,
            f"{prefix}_std": 0.0,
            f"{prefix}_min": 0.0,
            f"{prefix}_max": 0.0,
            f"{prefix}_median": 0.0,
            f"{prefix}_q25": 0.0,
            f"{prefix}_q75": 0.0,
            f"{prefix}_iqr": 0.0,
            f"{prefix}_range": 0.0,
            f"{prefix}_rms": 0.0,
            f"{prefix}_energy": 0.0,
            f"{prefix}_skew": 0.0,
            f"{prefix}_kurtosis": 0.0,
            f"{prefix}_mean_abs_diff": 0.0,
            f"{prefix}_zcr": 0.0,
        }

        if fs is not None:
            base.update(spectral_features(x, fs, prefix))

        return base

    dx = np.diff(x) if len(x) > 1 else np.array([0.0])
    std_v = np.std(x)

    features = {
        f"{prefix}_mean": clean_number(np.mean(x)),
        f"{prefix}_std": clean_number(std_v),
        f"{prefix}_min": clean_number(np.min(x)),
        f"{prefix}_max": clean_number(np.max(x)),
        f"{prefix}_median": clean_number(np.median(x)),
        f"{prefix}_q25": clean_number(np.percentile(x, 25)),
        f"{prefix}_q75": clean_number(np.percentile(x, 75)),
        f"{prefix}_iqr": clean_number(iqr(x)),
        f"{prefix}_range": clean_number(np.max(x) - np.min(x)),
        f"{prefix}_rms": clean_number(np.sqrt(np.mean(x ** 2))),
        f"{prefix}_energy": clean_number(np.mean(x ** 2)),
        f"{prefix}_skew": clean_number(skew(x) if len(x) > 2 and std_v > 1e-12 else 0.0),
        f"{prefix}_kurtosis": clean_number(kurtosis(x) if len(x) > 3 and std_v > 1e-12 else 0.0),
        f"{prefix}_mean_abs_diff": clean_number(np.mean(np.abs(dx)) if len(dx) > 0 else 0.0),
        f"{prefix}_zcr": clean_number(zero_crossing_rate(x)),
    }

    if fs is not None:
        features.update(spectral_features(x, fs, prefix))

    return features


def majority_label(labels):
    labels = np.asarray(labels).astype(int)
    labels = labels[np.isin(labels, [1, 2, 3, 4])]

    if len(labels) == 0:
        return None

    return Counter(labels).most_common(1)[0][0]


def label_to_binary(label):
    if label == 2:
        return 1

    if label in [1, 3, 4]:
        return 0

    return None


def extract_wrist_features(wrist, start, end):
    features = {}

    bvp = safe_array(wrist["BVP"]).reshape(-1)
    eda = safe_array(wrist["EDA"]).reshape(-1)
    temp = safe_array(wrist["TEMP"]).reshape(-1)
    acc = safe_array(wrist["ACC"])

    bvp_seg = bvp[int(start * FS_WRIST["BVP"]): int(end * FS_WRIST["BVP"])]
    eda_seg = eda[int(start * FS_WRIST["EDA"]): int(end * FS_WRIST["EDA"])]
    temp_seg = temp[int(start * FS_WRIST["TEMP"]): int(end * FS_WRIST["TEMP"])]
    acc_seg = acc[int(start * FS_WRIST["ACC"]): int(end * FS_WRIST["ACC"])]

    features.update(stats_features(bvp_seg, "w_bvp", fs=FS_WRIST["BVP"]))
    features.update(stats_features(eda_seg, "w_eda", fs=FS_WRIST["EDA"]))
    features.update(stats_features(temp_seg, "w_temp", fs=FS_WRIST["TEMP"]))

    if acc_seg.ndim == 1:
        acc_seg = acc_seg.reshape(-1, 3)

    if acc_seg.shape[1] >= 3:
        ax = acc_seg[:, 0]
        ay = acc_seg[:, 1]
        az = acc_seg[:, 2]
    else:
        ax = np.zeros(len(acc_seg))
        ay = np.zeros(len(acc_seg))
        az = np.zeros(len(acc_seg))

    amag = np.sqrt(ax ** 2 + ay ** 2 + az ** 2)

    features.update(stats_features(ax, "w_acc_x", fs=FS_WRIST["ACC"]))
    features.update(stats_features(ay, "w_acc_y", fs=FS_WRIST["ACC"]))
    features.update(stats_features(az, "w_acc_z", fs=FS_WRIST["ACC"]))
    features.update(stats_features(amag, "w_acc_mag", fs=FS_WRIST["ACC"]))

    return features


def extract_chest_features(chest, start, end):
    features = {}

    if chest is None:
        return features

    if "ACC" in chest:
        acc = safe_array(chest["ACC"])
        acc_seg = acc[int(start * FS_CHEST["ACC"]): int(end * FS_CHEST["ACC"])]

        if acc_seg.ndim == 1:
            acc_seg = acc_seg.reshape(-1, 3)

        if acc_seg.shape[1] >= 3:
            ax = acc_seg[:, 0]
            ay = acc_seg[:, 1]
            az = acc_seg[:, 2]
        else:
            ax = np.zeros(len(acc_seg))
            ay = np.zeros(len(acc_seg))
            az = np.zeros(len(acc_seg))

        amag = np.sqrt(ax ** 2 + ay ** 2 + az ** 2)

        features.update(stats_features(ax, "c_acc_x", fs=FS_CHEST["ACC"]))
        features.update(stats_features(ay, "c_acc_y", fs=FS_CHEST["ACC"]))
        features.update(stats_features(az, "c_acc_z", fs=FS_CHEST["ACC"]))
        features.update(stats_features(amag, "c_acc_mag", fs=FS_CHEST["ACC"]))

    for signal_name in ["ECG", "EDA", "EMG", "RESP", "TEMP"]:
        if signal_name in chest:
            sig = safe_array(chest[signal_name]).reshape(-1)
            seg = sig[int(start * FS_CHEST[signal_name]): int(end * FS_CHEST[signal_name])]

            prefix = "c_" + signal_name.lower()
            features.update(stats_features(seg, prefix, fs=FS_CHEST[signal_name]))

    return features


def process_subject(path):
    data = load_subject(path)

    subject_id = path.stem
    wrist = data["signal"].get("wrist", None)
    chest = data["signal"].get("chest", None)
    labels = np.asarray(data["label"]).astype(int)

    if wrist is None:
        return pd.DataFrame()

    bvp = safe_array(wrist["BVP"]).reshape(-1)
    eda = safe_array(wrist["EDA"]).reshape(-1)
    temp = safe_array(wrist["TEMP"]).reshape(-1)
    acc = safe_array(wrist["ACC"])

    duration_sec = min(
        len(bvp) / FS_WRIST["BVP"],
        len(eda) / FS_WRIST["EDA"],
        len(temp) / FS_WRIST["TEMP"],
        len(acc) / FS_WRIST["ACC"],
    )

    if FEATURE_MODE == "wrist_chest" and chest is not None:
        chest_durations = []

        for signal_name, fs in FS_CHEST.items():
            if signal_name in chest:
                chest_durations.append(len(chest[signal_name]) / fs)

        if len(chest_durations) > 0:
            duration_sec = min(duration_sec, min(chest_durations))

    label_fs = len(labels) / duration_sec

    rows = []
    start = 0

    while start + WINDOW_SEC <= duration_sec:
        end = start + WINDOW_SEC

        label_seg = labels[int(start * label_fs): int(end * label_fs)]
        original_label = majority_label(label_seg)

        if original_label is not None:
            target = label_to_binary(original_label)

            if target is not None:
                features = {}
                features.update(extract_wrist_features(wrist, start, end))

                if FEATURE_MODE == "wrist_chest":
                    features.update(extract_chest_features(chest, start, end))

                features["subject_id"] = subject_id
                features["wesad_label"] = int(original_label)
                features["target"] = int(target)
                features["window_start_sec"] = float(start)
                features["window_end_sec"] = float(end)

                rows.append(features)

        start += STEP_SEC

    return pd.DataFrame(rows)

In [ ]:
all_dfs = []

for path in tqdm(subject_files, desc="Processing WESAD subjects"):
    try:
        df_subject = process_subject(path)

        if len(df_subject) > 0:
            print(
                path.name,
                "windows:",
                len(df_subject),
                "class counts:",
                df_subject["target"].value_counts().to_dict()
            )

            all_dfs.append(df_subject)

    except Exception as e:
        print("Failed:", path.name)
        print("Error:", e)

if len(all_dfs) == 0:
    raise RuntimeError("No valid data created.")

df = pd.concat(all_dfs, ignore_index=True)
df = df.replace([np.inf, -np.inf], np.nan)

print("\nFinal dataset shape:", df.shape)
print("\nTarget counts:")
print(df["target"].value_counts())
print("\nSubjects:")
print(sorted(df["subject_id"].unique()))

processed_path = OUTPUT_DIR / "wesad_fast_processed_features.csv"
df.to_csv(processed_path, index=False)

print("\nProcessed features saved to:")
print(processed_path)

Processing WESAD subjects:   0%|          | 0/15 [00:00<?, ?it/s]

S10.pkl windows: 226 class counts: {0: 173, 1: 53}
S11.pkl windows: 222 class counts: {0: 173, 1: 49}
S13.pkl windows: 222 class counts: {0: 174, 1: 48}
S14.pkl windows: 220 class counts: {0: 171, 1: 49}
S15.pkl windows: 221 class counts: {0: 171, 1: 50}
S16.pkl windows: 220 class counts: {0: 171, 1: 49}
S17.pkl windows: 219 class counts: {0: 167, 1: 52}
S2.pkl windows: 212 class counts: {0: 167, 1: 45}
S3.pkl windows: 217 class counts: {0: 170, 1: 47}
S4.pkl windows: 217 class counts: {0: 170, 1: 47}
S5.pkl windows: 220 class counts: {0: 173, 1: 47}
S6.pkl windows: 219 class counts: {0: 171, 1: 48}
S7.pkl windows: 219 class counts: {0: 172, 1: 47}
S8.pkl windows: 220 class counts: {0: 171, 1: 49}
S9.pkl windows: 220 class counts: {0: 173, 1: 47}

Final dataset shape: (3294, 257)

Target counts:
target
0    2567
1     727
Name: count, dtype: int64

Subjects:
['S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']

Processed features saved to:
/

In [ ]:
metadata_cols = [
    "subject_id",
    "wesad_label",
    "target",
    "window_start_sec",
    "window_end_sec"
]

feature_cols = [c for c in df.columns if c not in metadata_cols]

X = df[feature_cols].copy()
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)

y = df["target"].astype(int)
groups = df["subject_id"]

print("Total NaN count:", int(X.isna().sum().sum()))


def valid_two_class_split(y_a, y_b):
    return len(np.unique(y_a)) == 2 and len(np.unique(y_b)) == 2


def find_group_split(X_data, y_data, groups_data, test_size, start_seed=1, end_seed=3000):
    for seed in range(start_seed, end_seed):
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=seed
        )

        idx_a, idx_b = next(splitter.split(X_data, y_data, groups_data))

        y_a = y_data.iloc[idx_a]
        y_b = y_data.iloc[idx_b]

        if valid_two_class_split(y_a, y_b):
            return idx_a, idx_b, seed

    raise RuntimeError("Could not create valid group split with both classes.")


if SPLIT_MODE == "random_split":
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=y
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y_train_val
    )

    groups_train = groups.loc[X_train.index]
    groups_val = groups.loc[X_val.index]
    groups_test = groups.loc[X_test.index]

else:
    train_val_idx, test_idx, seed_1 = find_group_split(
        X,
        y,
        groups,
        test_size=0.25,
        start_seed=1,
        end_seed=3000
    )

    X_train_val = X.iloc[train_val_idx]
    y_train_val = y.iloc[train_val_idx]
    groups_train_val = groups.iloc[train_val_idx]

    X_test = X.iloc[test_idx]
    y_test = y.iloc[test_idx]
    groups_test = groups.iloc[test_idx]

    train_idx, val_idx, seed_2 = find_group_split(
        X_train_val.reset_index(drop=True),
        y_train_val.reset_index(drop=True),
        groups_train_val.reset_index(drop=True),
        test_size=0.20,
        start_seed=3001,
        end_seed=6000
    )

    X_train = X_train_val.iloc[train_idx]
    y_train = y_train_val.iloc[train_idx]
    groups_train = groups_train_val.iloc[train_idx]

    X_val = X_train_val.iloc[val_idx]
    y_val = y_train_val.iloc[val_idx]
    groups_val = groups_train_val.iloc[val_idx]

    RANDOM_STATE = seed_1

train_subjects = sorted(groups_train.unique())
val_subjects = sorted(groups_val.unique())
test_subjects = sorted(groups_test.unique())

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

CLASS_WEIGHT = {
    int(cls): float(w)
    for cls, w in zip(classes, weights)
}

neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
SCALE_POS_WEIGHT = neg_count / max(pos_count, 1)

print("\nTrain shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

print("\nTrain subjects:", train_subjects)
print("Validation subjects:", val_subjects)
print("Test subjects:", test_subjects)

print("\nTrain class counts:", y_train.value_counts().to_dict())
print("Validation class counts:", y_val.value_counts().to_dict())
print("Test class counts:", y_test.value_counts().to_dict())

print("\nClass weights:", CLASS_WEIGHT)
print("Scale positive weight:", SCALE_POS_WEIGHT)

Total NaN count: 0

Train shape: (1761, 252)
Validation shape: (660, 252)
Test shape: (873, 252)

Train subjects: ['S10', 'S11', 'S15', 'S3', 'S4', 'S6', 'S7', 'S9']
Validation subjects: ['S16', 'S5', 'S8']
Test subjects: ['S13', 'S14', 'S17', 'S2']

Train class counts: {0: 1373, 1: 388}
Validation class counts: {0: 515, 1: 145}
Test class counts: {0: 679, 1: 194}

Class weights: {0: 0.6412964311726147, 1: 2.2693298969072164}
Scale positive weight: 3.538659793814433


In [ ]:
def get_positive_proba(model, X_data):
    if not hasattr(model, "predict_proba"):
        return None

    try:
        proba = model.predict_proba(X_data)
        proba = np.asarray(proba)

        if proba.ndim == 1:
            return proba

        if proba.ndim == 2 and proba.shape[1] == 1:
            return proba[:, 0]

        if proba.ndim == 2 and proba.shape[1] >= 2:
            return proba[:, 1]

        return None

    except Exception:
        return None


def metric_row(name, y_true, y_pred, p1=None, threshold=None, stage=None, model_type=None):
    acc = accuracy_score(y_true, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )

    try:
        auc = roc_auc_score(y_true, p1) if p1 is not None else None
    except Exception:
        auc = None

    return {
        "model": name,
        "model_type": model_type,
        "stage": stage,
        "threshold": None if threshold is None else float(threshold),
        "accuracy": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "roc_auc": None if auc is None else float(auc),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
    }


def evaluate_model(name, model, X_data, y_true, threshold=0.5, stage=None, model_type=None):
    p1 = get_positive_proba(model, X_data)

    if p1 is not None:
        y_pred = (p1 >= threshold).astype(int)
    else:
        y_pred = model.predict(X_data)
        y_pred = np.asarray(y_pred).reshape(-1)
        y_pred = (y_pred >= 0.5).astype(int)

    return metric_row(
        name=name,
        y_true=y_true,
        y_pred=y_pred,
        p1=p1,
        threshold=threshold,
        stage=stage,
        model_type=model_type
    )


def tune_threshold_on_validation(model, X_val_data, y_val_data):
    p1 = get_positive_proba(model, X_val_data)

    if p1 is None:
        return 0.5, None

    rows = []

    for threshold in np.arange(THRESHOLD_MIN, THRESHOLD_MAX + 1e-9, THRESHOLD_STEP):
        y_pred = (p1 >= threshold).astype(int)

        rows.append(metric_row(
            name="threshold_search",
            y_true=y_val_data,
            y_pred=y_pred,
            p1=p1,
            threshold=threshold,
            stage="validation"
        ))

    threshold_df = pd.DataFrame(rows)
    threshold_df["_auc_sort"] = threshold_df["roc_auc"].fillna(-1)

    threshold_df = threshold_df.sort_values(
        by=["accuracy", "f1_score", "_auc_sort"],
        ascending=False
    ).reset_index(drop=True)

    best = threshold_df.iloc[0]

    return float(best["threshold"]), best.to_dict()


def print_result(row):
    print("Threshold:", row.get("threshold"))
    print("Accuracy:", round(row["accuracy"], 5))
    print("Precision:", round(row["precision"], 5))
    print("Recall:", round(row["recall"], 5))
    print("F1-score:", round(row["f1_score"], 5))
    print("ROC-AUC:", None if row["roc_auc"] is None else round(row["roc_auc"], 5))
    print("Confusion matrix:", row["confusion_matrix"])

In [ ]:
ml_models = {}

ml_models["LogisticRegression"] = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("var", VarianceThreshold()),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=10,
        max_iter=3000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

ml_models["MLP"] = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("var", VarianceThreshold()),
    ("scaler", RobustScaler()),
    ("model", MLPClassifier(
        hidden_layer_sizes=(256, 256, 128, 64),
        activation="relu",
        alpha=0.0001,
        learning_rate_init=0.0002,
        max_iter=700,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        random_state=RANDOM_STATE
    ))
])

ml_models["ExtraTrees"] = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("var", VarianceThreshold()),
    ("model", ExtraTreesClassifier(
        n_estimators=700,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=1,
        max_features="log2",
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

ml_models["CatBoost"] = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("var", VarianceThreshold()),
    ("model", CatBoostClassifier(
        iterations=700,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5,
        loss_function="Logloss",
        eval_metric="Accuracy",
        class_weights=[1.0, SCALE_POS_WEIGHT],
        random_seed=RANDOM_STATE,
        verbose=False
    ))
])

ml_models["XGBoost"] = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("var", VarianceThreshold()),
    ("model", XGBClassifier(
        n_estimators=650,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.90,
        colsample_bytree=0.90,
        min_child_weight=1,
        gamma=0.05,
        reg_lambda=2.0,
        reg_alpha=0.1,
        scale_pos_weight=SCALE_POS_WEIGHT,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

ml_models["LightGBM"] = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("var", VarianceThreshold()),
    ("model", LGBMClassifier(
        n_estimators=700,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        subsample=0.90,
        subsample_freq=1,
        colsample_bytree=0.90,
        min_child_samples=10,
        reg_lambda=1.0,
        reg_alpha=0.1,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    ))
])

if RUN_SVM:
    ml_models["RBFSVM"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("var", VarianceThreshold()),
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            C=0.5,
            gamma=0.001,
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])

print("ML models:")
for name in ml_models:
    print("-", name)

ML models:
- LogisticRegression
- MLP
- ExtraTrees
- CatBoost
- XGBoost
- LightGBM
- RBFSVM


In [ ]:
results = []
trained_models = {}
trained_ml_models = {}
validation_thresholds = {}

if RUN_ML_MODELS:
    for name, model in ml_models.items():
        print("\n" + "=" * 80)
        print("Training ML:", name)
        print("=" * 80)

        try:
            model.fit(X_train, y_train)

            best_threshold, val_info = tune_threshold_on_validation(
                model,
                X_val,
                y_val
            )

            val_row = evaluate_model(
                name,
                model,
                X_val,
                y_val,
                threshold=best_threshold,
                stage="validation",
                model_type="machine_learning"
            )

            test_row = evaluate_model(
                name,
                model,
                X_test,
                y_test,
                threshold=best_threshold,
                stage="test",
                model_type="machine_learning"
            )

            test_row.update({
                "best_threshold": best_threshold,
                "selection_val_accuracy": val_row["accuracy"],
                "selection_val_f1_score": val_row["f1_score"],
                "selection_val_roc_auc": val_row["roc_auc"],
                "selection_val_confusion_matrix": val_row["confusion_matrix"],
            })

            results.append(test_row)
            trained_models[name] = model
            trained_ml_models[name] = model
            validation_thresholds[name] = best_threshold

            print("\nValidation result:")
            print_result(val_row)

            print("\nTest result:")
            print_result(test_row)

        except Exception as e:
            print("Failed:", name)
            print("Error:", e)


Training ML: LogisticRegression

Validation result:
Threshold: 0.8899999999999996
Accuracy: 0.95455
Precision: 0.84024
Recall: 0.97931
F1-score: 0.90446
ROC-AUC: 0.99772
Confusion matrix: [[488, 27], [3, 142]]

Test result:
Threshold: 0.8899999999999996
Accuracy: 0.87629
Precision: 0.65248
Recall: 0.94845
F1-score: 0.77311
ROC-AUC: 0.96912
Confusion matrix: [[581, 98], [10, 184]]

Training ML: MLP

Validation result:
Threshold: 0.7999999999999996
Accuracy: 0.96061
Precision: 0.91608
Recall: 0.90345
F1-score: 0.90972
ROC-AUC: 0.98402
Confusion matrix: [[503, 12], [14, 131]]

Test result:
Threshold: 0.7999999999999996
Accuracy: 0.87285
Precision: 0.92784
Recall: 0.46392
F1-score: 0.61856
ROC-AUC: 0.91305
Confusion matrix: [[672, 7], [104, 90]]

Training ML: ExtraTrees

Validation result:
Threshold: 0.34999999999999987
Accuracy: 0.99091
Precision: 0.96026
Recall: 1.0
F1-score: 0.97973
ROC-AUC: 0.9976
Confusion matrix: [[509, 6], [0, 145]]

Test result:
Threshold: 0.34999999999999987
Accu

In [ ]:
N_FEATURES = X_train.shape[1]

print("Number of features:", N_FEATURES)


def compile_binary_model(inputs, x, lr=0.001):
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


def build_deep_dnn(n_features, lr=0.001):
    inputs = layers.Input(shape=(n_features,))

    x = layers.Dense(256, activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.30)(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)

    return compile_binary_model(inputs, x, lr)


def build_tabular_cnn(n_features, lr=0.001):
    inputs = layers.Input(shape=(n_features,))
    x = layers.Reshape((n_features, 1))(inputs)

    x = layers.Conv1D(64, 5, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)

    avg_pool = layers.GlobalAveragePooling1D()(x)
    max_pool = layers.GlobalMaxPooling1D()(x)

    x = layers.Concatenate()([avg_pool, max_pool])
    x = layers.Dense(96, activation="relu")(x)
    x = layers.Dropout(0.25)(x)

    return compile_binary_model(inputs, x, lr)


def build_bilstm(n_features, lr=0.001):
    inputs = layers.Input(shape=(n_features,))
    x = layers.Reshape((n_features, 1))(inputs)

    x = layers.Bidirectional(
        layers.LSTM(48, return_sequences=True)
    )(x)

    x = layers.Bidirectional(
        layers.LSTM(24)
    )(x)

    x = layers.Dense(96, activation="relu")(x)
    x = layers.Dropout(0.25)(x)

    return compile_binary_model(inputs, x, lr)


def build_gru(n_features, lr=0.001):
    inputs = layers.Input(shape=(n_features,))
    x = layers.Reshape((n_features, 1))(inputs)

    x = layers.Bidirectional(
        layers.GRU(48, return_sequences=True)
    )(x)

    x = layers.Bidirectional(
        layers.GRU(24)
    )(x)

    x = layers.Dense(96, activation="relu")(x)
    x = layers.Dropout(0.25)(x)

    return compile_binary_model(inputs, x, lr)


def build_transformer_lite(n_features, lr=0.001):
    inputs = layers.Input(shape=(n_features,))

    x = layers.Reshape((n_features, 1))(inputs)
    x = layers.Dense(32)(x)

    attn = layers.MultiHeadAttention(
        num_heads=2,
        key_dim=16,
        dropout=0.10
    )(x, x)

    x = layers.LayerNormalization()(x + attn)

    ff = layers.Dense(64, activation="relu")(x)
    ff = layers.Dropout(0.10)(ff)
    ff = layers.Dense(32)(ff)

    x = layers.LayerNormalization()(x + ff)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(96, activation="relu")(x)
    x = layers.Dropout(0.25)(x)

    return compile_binary_model(inputs, x, lr)


class FastKerasStressClassifier:
    def __init__(
        self,
        build_fn,
        n_features,
        class_weight=None,
        epochs=70,
        batch_size=64,
        lr=0.001,
        random_state=42
    ):
        self.build_fn = build_fn
        self.n_features = n_features
        self.class_weight = class_weight
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.random_state = random_state

        self.imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()
        self.model = None
        self.classes_ = np.array([0, 1])

    def fit(self, X, y, X_val=None, y_val=None):
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(self.random_state)

        X_imp = self.imputer.fit_transform(X)
        X_scaled = self.scaler.fit_transform(X_imp)

        y_arr = np.asarray(y).astype(int)

        self.model = self.build_fn(
            n_features=self.n_features,
            lr=self.lr
        )

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_accuracy",
                patience=DL_PATIENCE,
                restore_best_weights=True,
                mode="max"
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=max(3, DL_PATIENCE // 2),
                min_lr=1e-5
            )
        ]

        if X_val is not None and y_val is not None:
            X_val_imp = self.imputer.transform(X_val)
            X_val_scaled = self.scaler.transform(X_val_imp)

            validation_data = (X_val_scaled, np.asarray(y_val).astype(int))
            validation_split = 0.0
        else:
            validation_data = None
            validation_split = 0.15

        self.model.fit(
            X_scaled,
            y_arr,
            epochs=self.epochs,
            batch_size=self.batch_size,
            validation_data=validation_data,
            validation_split=validation_split,
            class_weight=self.class_weight,
            callbacks=callbacks,
            verbose=0
        )

        return self

    def predict_proba(self, X):
        X_imp = self.imputer.transform(X)
        X_scaled = self.scaler.transform(X_imp)

        p1 = self.model.predict(X_scaled, verbose=0).reshape(-1)
        p1 = np.clip(p1, 0.0, 1.0)

        return np.vstack([1.0 - p1, p1]).T

    def predict(self, X):
        p1 = self.predict_proba(X)[:, 1]
        return (p1 >= 0.5).astype(int)


dl_builders = {
    "DeepDNN": build_deep_dnn,
    "TabularCNN": build_tabular_cnn,
    "BiLSTM": build_bilstm,
    "GRU": build_gru,
}

if RUN_TRANSFORMER:
    dl_builders["TransformerLite"] = build_transformer_lite

print("Deep models:")
for name in dl_builders:
    print("-", name)

Number of features: 252
Deep models:
- DeepDNN
- TabularCNN
- BiLSTM
- GRU
- TransformerLite


In [ ]:
trained_dl_models = {}

if RUN_DEEP_MODELS:
    for idx, (name, builder) in enumerate(dl_builders.items(), start=1):
        print("\n" + "=" * 80)
        print("Training Deep Learning:", name)
        print("=" * 80)

        try:
            model = FastKerasStressClassifier(
                build_fn=builder,
                n_features=N_FEATURES,
                class_weight=CLASS_WEIGHT,
                epochs=DL_EPOCHS,
                batch_size=DL_BATCH_SIZE,
                lr=0.001,
                random_state=RANDOM_STATE + idx
            )

            model.fit(
                X_train,
                y_train,
                X_val=X_val,
                y_val=y_val
            )

            best_threshold, val_info = tune_threshold_on_validation(
                model,
                X_val,
                y_val
            )

            val_row = evaluate_model(
                name,
                model,
                X_val,
                y_val,
                threshold=best_threshold,
                stage="validation",
                model_type="deep_learning"
            )

            test_row = evaluate_model(
                name,
                model,
                X_test,
                y_test,
                threshold=best_threshold,
                stage="test",
                model_type="deep_learning"
            )

            test_row.update({
                "best_threshold": best_threshold,
                "selection_val_accuracy": val_row["accuracy"],
                "selection_val_f1_score": val_row["f1_score"],
                "selection_val_roc_auc": val_row["roc_auc"],
                "selection_val_confusion_matrix": val_row["confusion_matrix"],
            })

            results.append(test_row)
            trained_models[name] = model
            trained_dl_models[name] = model
            validation_thresholds[name] = best_threshold

            print("\nValidation result:")
            print_result(val_row)

            print("\nTest result:")
            print_result(test_row)

        except Exception as e:
            print("Failed deep model:", name)
            print("Error:", e)


Training Deep Learning: DeepDNN

Validation result:
Threshold: 0.8799999999999996
Accuracy: 0.97879
Precision: 0.92258
Recall: 0.98621
F1-score: 0.95333
ROC-AUC: 0.99792
Confusion matrix: [[503, 12], [2, 143]]

Test result:
Threshold: 0.8799999999999996
Accuracy: 0.93242
Precision: 0.92994
Recall: 0.75258
F1-score: 0.83191
ROC-AUC: 0.97747
Confusion matrix: [[668, 11], [48, 146]]

Training Deep Learning: TabularCNN

Validation result:
Threshold: 0.5799999999999997
Accuracy: 0.91515
Precision: 0.80272
Recall: 0.81379
F1-score: 0.80822
ROC-AUC: 0.95571
Confusion matrix: [[486, 29], [27, 118]]

Test result:
Threshold: 0.5799999999999997
Accuracy: 0.89576
Precision: 0.86525
Recall: 0.62887
F1-score: 0.72836
ROC-AUC: 0.86863
Confusion matrix: [[660, 19], [72, 122]]

Training Deep Learning: BiLSTM

Validation result:
Threshold: 0.6899999999999997
Accuracy: 0.90758
Precision: 0.78
Recall: 0.8069
F1-score: 0.79322
ROC-AUC: 0.94295
Confusion matrix: [[482, 33], [28, 117]]

Test result:
Thresho

In [ ]:
class FastSoftVotingClassifier:
    def __init__(self, estimators):
        self.estimators = estimators
        self.classes_ = np.array([0, 1])

    def predict_proba(self, X_data):
        probs = []

        for name, model in self.estimators:
            p1 = get_positive_proba(model, X_data)

            if p1 is not None:
                probs.append(np.asarray(p1).reshape(-1))

        if len(probs) == 0:
            raise RuntimeError("No model produced probability.")

        p1_mean = np.mean(np.vstack(probs), axis=0)
        p1_mean = np.clip(p1_mean, 0.0, 1.0)

        return np.vstack([1.0 - p1_mean, p1_mean]).T

    def predict(self, X_data):
        p1 = self.predict_proba(X_data)[:, 1]
        return (p1 >= 0.5).astype(int)


if RUN_ENSEMBLE and len(trained_ml_models) >= 3:
    temp_df = pd.DataFrame(results).copy()
    temp_df = temp_df[temp_df["model"].isin(trained_ml_models.keys())].copy()
    temp_df["_val_auc_sort"] = temp_df["selection_val_roc_auc"].fillna(-1)

    temp_df = temp_df.sort_values(
        by=["selection_val_accuracy", "selection_val_f1_score", "_val_auc_sort"],
        ascending=False
    ).reset_index(drop=True)

    top_ml_names = temp_df["model"].head(5).tolist()

    print("Top ML models for ensemble:")
    print(top_ml_names)

    ensemble = FastSoftVotingClassifier(
        estimators=[(name, trained_ml_models[name]) for name in top_ml_names]
    )

    best_threshold, val_info = tune_threshold_on_validation(
        ensemble,
        X_val,
        y_val
    )

    val_row = evaluate_model(
        "SoftVotingML_Top5",
        ensemble,
        X_val,
        y_val,
        threshold=best_threshold,
        stage="validation",
        model_type="ensemble"
    )

    test_row = evaluate_model(
        "SoftVotingML_Top5",
        ensemble,
        X_test,
        y_test,
        threshold=best_threshold,
        stage="test",
        model_type="ensemble"
    )

    test_row.update({
        "best_threshold": best_threshold,
        "selection_val_accuracy": val_row["accuracy"],
        "selection_val_f1_score": val_row["f1_score"],
        "selection_val_roc_auc": val_row["roc_auc"],
        "selection_val_confusion_matrix": val_row["confusion_matrix"],
        "ensemble_members": top_ml_names
    })

    results.append(test_row)
    trained_models["SoftVotingML_Top5"] = ensemble
    validation_thresholds["SoftVotingML_Top5"] = best_threshold

    print("\nValidation result:")
    print_result(val_row)

    print("\nTest result:")
    print_result(test_row)

Top ML models for ensemble:
['CatBoost', 'XGBoost', 'LightGBM', 'ExtraTrees', 'RBFSVM']

Validation result:
Threshold: 0.43999999999999984
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0
ROC-AUC: 1.0
Confusion matrix: [[515, 0], [0, 145]]

Test result:
Threshold: 0.43999999999999984
Accuracy: 0.8866
Precision: 0.69388
Recall: 0.87629
F1-score: 0.77449
ROC-AUC: 0.95952
Confusion matrix: [[604, 75], [24, 170]]


In [ ]:
results_df = pd.DataFrame(results).copy()

results_df["_val_auc_sort"] = results_df["selection_val_roc_auc"].fillna(-1)
results_df["_test_auc_sort"] = results_df["roc_auc"].fillna(-1)

selection_df = results_df.sort_values(
    by=["selection_val_accuracy", "selection_val_f1_score", "_val_auc_sort"],
    ascending=False
).reset_index(drop=True)

test_ranking_df = results_df.sort_values(
    by=["accuracy", "f1_score", "_test_auc_sort"],
    ascending=False
).reset_index(drop=True)

selection_df = selection_df.drop(columns=["_val_auc_sort", "_test_auc_sort"], errors="ignore")
test_ranking_df = test_ranking_df.drop(columns=["_val_auc_sort", "_test_auc_sort"], errors="ignore")

print("\n" + "=" * 100)
print("FINAL MODEL SELECTION - SELECTED BY VALIDATION")
print("=" * 100)

display_cols = [
    "model",
    "model_type",
    "best_threshold",
    "selection_val_accuracy",
    "selection_val_f1_score",
    "selection_val_roc_auc",
    "accuracy",
    "precision",
    "recall",
    "f1_score",
    "roc_auc",
    "confusion_matrix"
]

existing_cols = [c for c in display_cols if c in selection_df.columns]
print(selection_df[existing_cols].to_string(index=False))

print("\n" + "=" * 100)
print("TEST RANKING - REPORTING ONLY")
print("=" * 100)

existing_cols = [c for c in display_cols if c in test_ranking_df.columns]
print(test_ranking_df[existing_cols].to_string(index=False))

best_model_name = selection_df.iloc[0]["model"]
best_model = trained_models[best_model_name]
best_threshold = float(selection_df.iloc[0]["best_threshold"])

print("\nSelected best model by validation:")
print(best_model_name)

print("\nSelected best threshold:")
print(best_threshold)

print("\nSelected model test accuracy:")
print(float(selection_df.iloc[0]["accuracy"]))

p1_best = get_positive_proba(best_model, X_test)

if p1_best is not None:
    y_pred_best = (p1_best >= best_threshold).astype(int)
else:
    y_pred_best = best_model.predict(X_test)
    y_pred_best = np.asarray(y_pred_best).reshape(-1)
    y_pred_best = (y_pred_best >= 0.5).astype(int)

report = classification_report(
    y_test,
    y_pred_best,
    target_names=["non_stress", "stress"],
    zero_division=0
)

print("\nFinal Classification Report:")
print(report)


FINAL MODEL SELECTION - SELECTED BY VALIDATION
             model       model_type  best_threshold  selection_val_accuracy  selection_val_f1_score  selection_val_roc_auc  accuracy  precision   recall  f1_score  roc_auc        confusion_matrix
          CatBoost machine_learning            0.41                1.000000                1.000000               1.000000  0.844215   0.609848 0.829897  0.703057 0.922475 [[576, 103], [33, 161]]
 SoftVotingML_Top5         ensemble            0.44                1.000000                1.000000               1.000000  0.886598   0.693878 0.876289  0.774487 0.959522  [[604, 75], [24, 170]]
           XGBoost machine_learning            0.21                0.998485                0.996564               0.999920  0.872852   0.647687 0.938144  0.766316 0.954329  [[580, 99], [12, 182]]
          LightGBM machine_learning            0.27                0.996970                0.993151               0.999879  0.861397   0.647773 0.824742  0.725624 0.940

In [ ]:
# ============================================================
# SAVE VS CODE READY PACKAGE - COMPLETE FIXED CELL
# ============================================================

import json
import shutil
import joblib
from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1) Choose model to save for VS Code project
# ------------------------------------------------------------
# DeepDNN had the best test result in your run.
# Change this to "RBFSVM" if you want a lighter model without TensorFlow.
DEPLOYMENT_MODEL_NAME = "DeepDNN"

if DEPLOYMENT_MODEL_NAME in trained_models:
    best_model_name = DEPLOYMENT_MODEL_NAME
    best_model = trained_models[best_model_name]

    deploy_row = results_df[results_df["model"] == best_model_name].iloc[0]
    best_threshold = float(deploy_row["best_threshold"])

    print("\n" + "=" * 80)
    print("DEPLOYMENT MODEL OVERRIDE")
    print("=" * 80)
    print("Model selected for VS Code package:", best_model_name)
    print("Threshold:", best_threshold)
    print("Test accuracy:", float(deploy_row["accuracy"]))
    print("Test precision:", float(deploy_row["precision"]))
    print("Test recall:", float(deploy_row["recall"]))
    print("Test F1:", float(deploy_row["f1_score"]))
    print("Test ROC-AUC:", float(deploy_row["roc_auc"]))
    print("Confusion matrix:", deploy_row["confusion_matrix"])

else:
    print("\nRequested deployment model was not found:", DEPLOYMENT_MODEL_NAME)
    print("Using currently selected model instead:", best_model_name)


# ------------------------------------------------------------
# 2) Rebuild final report for selected deployment model
# ------------------------------------------------------------
p1_deploy = get_positive_proba(best_model, X_test)

if p1_deploy is not None:
    y_pred_deploy = (p1_deploy >= best_threshold).astype(int)
else:
    y_pred_deploy = best_model.predict(X_test)
    y_pred_deploy = np.asarray(y_pred_deploy).reshape(-1)
    y_pred_deploy = (y_pred_deploy >= 0.5).astype(int)

deployment_report = classification_report(
    y_test,
    y_pred_deploy,
    target_names=["non_stress", "stress"],
    zero_division=0
)

print("\nDeployment Classification Report:")
print(deployment_report)


# ------------------------------------------------------------
# 3) Create package folders
# ------------------------------------------------------------
VSCODE_PACKAGE_DIR = OUTPUT_DIR / "vscode_model_package"
VSCODE_MODELS_DIR = VSCODE_PACKAGE_DIR / "models"

if VSCODE_PACKAGE_DIR.exists():
    shutil.rmtree(VSCODE_PACKAGE_DIR)

VSCODE_PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
VSCODE_MODELS_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 4) Helper functions
# ------------------------------------------------------------
def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}

    if isinstance(obj, list):
        return [make_json_safe(x) for x in obj]

    if isinstance(obj, tuple):
        return [make_json_safe(x) for x in obj]

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, (np.integer,)):
        return int(obj)

    if isinstance(obj, (np.floating,)):
        return float(obj)

    if isinstance(obj, (np.bool_,)):
        return bool(obj)

    try:
        json.dumps(obj)
        return obj
    except Exception:
        return str(obj)


def safe_name(name):
    return "".join(
        ch if ch.isalnum() or ch in ["_", "-"] else "_"
        for ch in str(name)
    )


def save_model_spec(model, model_name, save_dir):
    """
    Saves:
    - sklearn / XGBoost / LightGBM / CatBoost pipeline as .joblib
    - FastKerasStressClassifier as .keras + preprocessor.joblib
    - FastSoftVotingClassifier by saving each member model recursively
    """

    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    clean = safe_name(model_name)

    # Deep learning model
    if "FastKerasStressClassifier" in globals() and isinstance(model, FastKerasStressClassifier):
        keras_path = save_dir / f"{clean}.keras"
        preprocessor_path = save_dir / f"{clean}_preprocessor.joblib"

        model.model.save(str(keras_path), overwrite=True)

        preprocessor_payload = {
            "imputer": model.imputer,
            "scaler": model.scaler,
            "n_features": model.n_features,
            "class_name": "FastKerasStressClassifier"
        }

        joblib.dump(preprocessor_payload, preprocessor_path)

        return {
            "type": "keras_with_preprocessor",
            "keras_model_path": str(keras_path.relative_to(VSCODE_PACKAGE_DIR)),
            "preprocessor_path": str(preprocessor_path.relative_to(VSCODE_PACKAGE_DIR)),
            "model_name": model_name
        }

    # Soft voting ensemble
    if "FastSoftVotingClassifier" in globals() and isinstance(model, FastSoftVotingClassifier):
        members = []

        members_dir = save_dir / f"{clean}_members"
        members_dir.mkdir(parents=True, exist_ok=True)

        for member_name, member_model in model.estimators:
            member_save_dir = members_dir / safe_name(member_name)

            member_spec = save_model_spec(
                member_model,
                member_name,
                member_save_dir
            )

            members.append({
                "name": member_name,
                "spec": member_spec
            })

        return {
            "type": "soft_voting",
            "model_name": model_name,
            "members": members
        }

    # Normal ML model / pipeline
    model_path = save_dir / f"{clean}.joblib"
    joblib.dump(model, model_path)

    return {
        "type": "joblib_model",
        "model_path": str(model_path.relative_to(VSCODE_PACKAGE_DIR)),
        "model_name": model_name
    }


# ------------------------------------------------------------
# 5) Save selected model
# ------------------------------------------------------------
best_model_spec = save_model_spec(
    best_model,
    best_model_name,
    VSCODE_MODELS_DIR / safe_name(best_model_name)
)

model_spec_path = VSCODE_MODELS_DIR / "model_spec.json"

with open(model_spec_path, "w") as f:
    json.dump(make_json_safe(best_model_spec), f, indent=2)


# ------------------------------------------------------------
# 6) Save feature names
# ------------------------------------------------------------
feature_names_path = VSCODE_PACKAGE_DIR / "feature_names.json"

with open(feature_names_path, "w") as f:
    json.dump(list(feature_cols), f, indent=2)


# ------------------------------------------------------------
# 7) Save metadata
# ------------------------------------------------------------
metadata = {
    "project_name": "AI Health Monitoring Bracelet",
    "task": "binary_stress_detection",
    "dataset": "WESAD",
    "feature_mode": FEATURE_MODE,
    "split_mode": SPLIT_MODE,
    "best_model_name": best_model_name,
    "best_threshold": float(best_threshold),
    "window_sec": WINDOW_SEC,
    "step_sec": STEP_SEC,
    "label_mapping": {
        "0": "non_stress",
        "1": "stress"
    },
    "original_wesad_label_mapping": {
        "1": "baseline",
        "2": "stress",
        "3": "amusement",
        "4": "meditation"
    },
    "train_subjects": train_subjects,
    "validation_subjects": val_subjects,
    "test_subjects": test_subjects,
    "model_spec": best_model_spec,
    "metrics_selection_by_validation": selection_df.to_dict(orient="records"),
    "metrics_test_ranking": test_ranking_df.to_dict(orient="records")
}

metadata_path = VSCODE_PACKAGE_DIR / "metadata.json"

with open(metadata_path, "w") as f:
    json.dump(make_json_safe(metadata), f, indent=2)


# ------------------------------------------------------------
# 8) Save sample input
# ------------------------------------------------------------
sample_features = X_test.iloc[0][feature_cols].to_dict()
sample_input_path = VSCODE_PACKAGE_DIR / "sample_input.json"

with open(sample_input_path, "w") as f:
    json.dump(make_json_safe(sample_features), f, indent=2)


# ------------------------------------------------------------
# 9) Save reports
# ------------------------------------------------------------
selection_df.to_csv(
    VSCODE_PACKAGE_DIR / "model_selection_by_validation.csv",
    index=False
)

test_ranking_df.to_csv(
    VSCODE_PACKAGE_DIR / "test_ranking_for_reporting.csv",
    index=False
)

with open(VSCODE_PACKAGE_DIR / "classification_report.txt", "w") as f:
    f.write("Model saved for VS Code deployment: " + str(best_model_name) + "\n")
    f.write("Threshold: " + str(best_threshold) + "\n\n")
    f.write(deployment_report)


# ------------------------------------------------------------
# 10) Save inference.py
# ------------------------------------------------------------
inference_code = r'''
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

try:
    import tensorflow as tf
except Exception:
    tf = None


def get_positive_proba(model, X_data):
    if not hasattr(model, "predict_proba"):
        return None

    proba = model.predict_proba(X_data)
    proba = np.asarray(proba)

    if proba.ndim == 1:
        return proba

    if proba.ndim == 2 and proba.shape[1] == 1:
        return proba[:, 0]

    if proba.ndim == 2 and proba.shape[1] >= 2:
        return proba[:, 1]

    return None


class LoadedKerasWithPreprocessor:
    def __init__(self, keras_model, imputer, scaler):
        self.keras_model = keras_model
        self.imputer = imputer
        self.scaler = scaler
        self.classes_ = np.array([0, 1])

    def predict_proba(self, X):
        X_imp = self.imputer.transform(X)
        X_scaled = self.scaler.transform(X_imp)

        p1 = self.keras_model.predict(X_scaled, verbose=0).reshape(-1)
        p1 = np.clip(p1, 0.0, 1.0)

        return np.vstack([1.0 - p1, p1]).T

    def predict(self, X):
        p1 = self.predict_proba(X)[:, 1]
        return (p1 >= 0.5).astype(int)


class LoadedSoftVoting:
    def __init__(self, estimators):
        self.estimators = estimators
        self.classes_ = np.array([0, 1])

    def predict_proba(self, X):
        probs = []

        for name, model in self.estimators:
            p1 = get_positive_proba(model, X)

            if p1 is not None:
                probs.append(np.asarray(p1).reshape(-1))

        if len(probs) == 0:
            raise RuntimeError("No model produced probabilities.")

        p1_mean = np.mean(np.vstack(probs), axis=0)
        p1_mean = np.clip(p1_mean, 0.0, 1.0)

        return np.vstack([1.0 - p1_mean, p1_mean]).T

    def predict(self, X):
        p1 = self.predict_proba(X)[:, 1]
        return (p1 >= 0.5).astype(int)


def load_model_from_spec(spec, package_dir):
    package_dir = Path(package_dir)

    if spec["type"] == "joblib_model":
        return joblib.load(package_dir / spec["model_path"])

    if spec["type"] == "keras_with_preprocessor":
        if tf is None:
            raise RuntimeError("TensorFlow is required to load this Keras model.")

        keras_model = tf.keras.models.load_model(
            package_dir / spec["keras_model_path"],
            compile=False
        )

        prep = joblib.load(package_dir / spec["preprocessor_path"])

        return LoadedKerasWithPreprocessor(
            keras_model=keras_model,
            imputer=prep["imputer"],
            scaler=prep["scaler"]
        )

    if spec["type"] == "soft_voting":
        members = []

        for member in spec["members"]:
            member_model = load_model_from_spec(
                member["spec"],
                package_dir
            )

            members.append((member["name"], member_model))

        return LoadedSoftVoting(members)

    raise ValueError("Unknown model spec type: " + str(spec["type"]))


class WESADStressPredictor:
    def __init__(self, package_dir):
        self.package_dir = Path(package_dir)

        with open(self.package_dir / "metadata.json", "r") as f:
            self.metadata = json.load(f)

        with open(self.package_dir / "feature_names.json", "r") as f:
            self.feature_names = json.load(f)

        with open(self.package_dir / "models" / "model_spec.json", "r") as f:
            self.model_spec = json.load(f)

        self.model = load_model_from_spec(
            self.model_spec,
            self.package_dir
        )

        self.threshold = float(self.metadata["best_threshold"])
        self.label_mapping = self.metadata["label_mapping"]
        self.model_name = self.metadata["best_model_name"]

    def predict_from_features(self, features):
        row = pd.DataFrame([features])
        row = row.reindex(columns=self.feature_names, fill_value=0.0)
        row = row.replace([np.inf, -np.inf], np.nan)

        p1 = get_positive_proba(self.model, row)

        if p1 is not None:
            stress_probability = float(np.asarray(p1).reshape(-1)[0])
            stress_probability = float(np.clip(stress_probability, 0.0, 1.0))

            pred_id = int(stress_probability >= self.threshold)

            return {
                "prediction": self.label_mapping[str(pred_id)],
                "prediction_id": pred_id,
                "stress_probability": stress_probability,
                "non_stress_probability": float(1.0 - stress_probability),
                "confidence": float(max(stress_probability, 1.0 - stress_probability)),
                "threshold": self.threshold,
                "model_name": self.model_name
            }

        pred_raw = self.model.predict(row)
        pred_id = int(np.asarray(pred_raw).reshape(-1)[0])

        return {
            "prediction": self.label_mapping[str(pred_id)],
            "prediction_id": pred_id,
            "stress_probability": None,
            "non_stress_probability": None,
            "confidence": None,
            "threshold": self.threshold,
            "model_name": self.model_name
        }

    def predict_from_json(self, json_path):
        with open(json_path, "r") as f:
            features = json.load(f)

        return self.predict_from_features(features)

    def predict_from_csv_row(self, csv_path, row_index=0):
        df = pd.read_csv(csv_path)
        features = df.iloc[row_index].to_dict()

        return self.predict_from_features(features)
'''

with open(VSCODE_PACKAGE_DIR / "inference.py", "w") as f:
    f.write(inference_code)


# ------------------------------------------------------------
# 11) Save example_predict.py
# ------------------------------------------------------------
example_code = r'''
from inference import WESADStressPredictor

predictor = WESADStressPredictor(".")

result = predictor.predict_from_json("sample_input.json")

print(result)
'''

with open(VSCODE_PACKAGE_DIR / "example_predict.py", "w") as f:
    f.write(example_code)


# ------------------------------------------------------------
# 12) Save requirements.txt
# ------------------------------------------------------------
requirements_text = """
numpy
pandas
scipy
scikit-learn
joblib
catboost
xgboost
lightgbm
tensorflow
"""

with open(VSCODE_PACKAGE_DIR / "requirements.txt", "w") as f:
    f.write(requirements_text.strip() + "\n")


# ------------------------------------------------------------
# 13) Save README.md
# ------------------------------------------------------------
readme_lines = [
    "# WESAD Stress Detection Model Package",
    "",
    "This folder is ready to copy into your VS Code project.",
    "",
    "## Best model",
    str(best_model_name),
    "",
    "## Best threshold",
    str(best_threshold),
    "",
    "## Feature mode",
    str(FEATURE_MODE),
    "",
    "## Split mode",
    str(SPLIT_MODE),
    "",
    "## Files",
    "- inference.py: use this in your backend/project to load the model and predict.",
    "- metadata.json: model information, threshold, labels, metrics.",
    "- feature_names.json: expected feature names and order.",
    "- sample_input.json: example input feature dictionary.",
    "- models/: saved model files.",
    "- requirements.txt: Python dependencies.",
    "- example_predict.py: quick test script.",
    "- model_selection_by_validation.csv: validation ranking.",
    "- test_ranking_for_reporting.csv: test ranking.",
    "",
    "## Setup in VS Code",
    "pip install -r requirements.txt",
    "python example_predict.py",
    "",
    "## Usage",
    "from inference import WESADStressPredictor",
    'predictor = WESADStressPredictor("path/to/vscode_model_package")',
    'result = predictor.predict_from_json("path/to/vscode_model_package/sample_input.json")',
    "print(result)",
    "",
    "## Important",
    "The input must be extracted features, not raw sensor signals.",
    "",
    "For bracelet-only deployment, retrain using:",
    'FEATURE_MODE = "wrist"',
]

readme_text = "\n".join(readme_lines)

with open(VSCODE_PACKAGE_DIR / "README.md", "w") as f:
    f.write(readme_text)


# ------------------------------------------------------------
# 14) Zip package
# ------------------------------------------------------------
zip_base = BASE_DIR / "wesad_vscode_model_package"
zip_file = Path(str(zip_base) + ".zip")

if zip_file.exists():
    zip_file.unlink()

shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(VSCODE_PACKAGE_DIR)
)

print("\nVS Code package created:")
print(zip_file)

print("\nPackage folder:")
print(VSCODE_PACKAGE_DIR)

try:
    from google.colab import files
    files.download(str(zip_file))
except Exception as e:
    print("Download manually from:")
    print(zip_file)
    print("Download skipped:", e)


DEPLOYMENT MODEL OVERRIDE
Model selected for VS Code package: DeepDNN
Threshold: 0.8799999999999996
Test accuracy: 0.9324169530355098
Test precision: 0.9299363057324841
Test recall: 0.7525773195876289
Test F1: 0.8319088319088319
Test ROC-AUC: 0.9774683813370179
Confusion matrix: [[668, 11], [48, 146]]

Deployment Classification Report:
              precision    recall  f1-score   support

  non_stress       0.93      0.98      0.96       679
      stress       0.93      0.75      0.83       194

    accuracy                           0.93       873
   macro avg       0.93      0.87      0.89       873
weighted avg       0.93      0.93      0.93       873


VS Code package created:
/content/wesad_vscode_model_package.zip

Package folder:
/content/wesad_fast_output/vscode_model_package


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>